# Задача: Поиск и оптимизация наилучших параметров моделей машинного обучения (LogisticRegresiion, RandomForestClassifier) с помощью различных алгоритмов (RandomizedSearch, GridSearch, Optuna, Hyperopt)

In [1]:
import warnings #для пропуска предупреждений
warnings.filterwarnings('ignore')

In [ ]:
#импорт библиотек
import numpy as np #для матричных вычислений
import pandas as pd #для анализа и предобработки данных
import matplotlib.pyplot as plt #для визуализации
import seaborn as sns #для визуализации
from sklearn import linear_model,metrics, tree, ensemble, preprocessing
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV, cross_val_score, train_test_split 
import hyperopt #модель для подбора гиперпараметров
from hyperopt import hp, fmin, tpe, Trials # fmin - основная функция, она будет минимизировать наш функционал, tpe - алгоритм оптимизации
# hp - включает набор методов для объявления пространства поиска гиперпараметров, trails - используется для логирования результатов
import optuna #модель для подбора гиперпараметров

%matplotlib inline
plt.style.use('seaborn-v0_8')


In [4]:
#читаем файл
data = pd.read_csv('D:/IDE/data/_train_sem09.csv') 
data.head()

,Activity,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,D1767,D1768,D1769,D1770,D1771,D1772,D1773,D1774,D1775,D1776
0,1,0.000000,0.497009,0.10,0.0,0.132956,0.678031,0.273166,0.585445,0.743663,...,0,0,0,0,0,0,0,0,0,0
1,1,0.366667,0.606291,0.05,0.0,0.111209,0.803455,0.106105,0.411754,0.836582,...,1,1,1,1,0,1,0,0,1,0
2,1,0.033300,0.480124,0.00,0.0,0.209791,0.610350,0.356453,0.517720,0.679051,...,0,0,0,0,0,0,0,0,0,0
3,1,0.000000,0.538825,0.00,0.5,0.196344,0.724230,0.235606,0.288764,0.805110,...,0,0,0,0,0,0,0,0,0,0
4,0,0.100000,0.517794,0.00,0.0,0.494734,0.781422,0.154361,0.303809,0.812646,...,0,0,0,0,0,0,0,0,0,0


In [7]:
#матрицы Х и матрица у с ответами
X = data.drop('Activity',axis=1)
y = data['Activity']

In [8]:
#делим выборки на тестовую и тренировочную
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state = 42, test_size = 0.2)

In [9]:
#Создаем объект класса логистическая регрессия
log_reg = linear_model.LogisticRegression(max_iter = 50,random_state=42)
#обучаем модель
log_reg.fit(X_train,y_train)
#предсказываем результат
y_pred_test_log = log_reg.predict(X_test)
y_pred_train_log = log_reg.predict(X_train)

print('F1 на тренирвочной {:.2f}'.format(metrics.f1_score(y_train, y_pred_train_log)))
print('F1 на тестовой {:.2f}'.format(metrics.f1_score(y_test, y_pred_test_log)))

F1 на тренирвочной 0.88
F1 на тестовой 0.78


In [10]:
#Создаем объект класса случайный лес
ran_tr = ensemble.RandomForestClassifier(random_state=42)
#обучаем модель
ran_tr.fit(X_train,y_train)
#предсказываем результат
y_pred_test_tr = ran_tr.predict(X_test)
y_pred_train_tr = ran_tr.predict(X_train)

print('F1 на тренирвочной {:.2f}'.format(metrics.f1_score(y_train, y_pred_train_tr)))
print('F1 на тестовой {:.2f}'.format(metrics.f1_score(y_test, y_pred_test_tr)))

F1 на тренирвочной 1.00
F1 на тестовой 0.81


# <center> GridSearchCV + LogisticRegression

In [ ]:
#задаем параметры
param_grid = [
              {'penalty': ['l2', 'none'] , # тип регуляризации
              'solver': ['lbfgs', 'sag'], # алгоритм оптимизации
               'C': [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 1]}, # уровень силы регурялизации
              
              {'penalty': ['l1', 'l2'] ,
              'solver': ['liblinear', 'saga'],
               'C': [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 1]}
]
#создаем объект gridsearch с логистической регрессией для перебора параметров
grid_search = GridSearchCV(
    estimator=linear_model.LogisticRegression(random_state=42, max_iter=50), 
    param_grid=param_grid, 
    cv=5, 
    n_jobs = -1
)  
#замер времени
%time grid_search.fit(X_train, y_train) 
#предсказание результата
y_test_pred = grid_search.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(grid_search.best_params_))

CPU times: total: 3.55 s
Wall time: 35.1 s
f1_score на тестовом наборе: 0.79
Наилучшие значения гиперпараметров: {'C': 0.3, 'penalty': 'l1', 'solver': 'saga'}


Метрику удалось улучшить, но незначительно

# <center> RandomizedSearchCV + LogisticRegression

In [ ]:
#создаем RandomizedSearch с линейной регрессией
random_search = RandomizedSearchCV(
    estimator=linear_model.LogisticRegression(random_state=42, max_iter=50), 
    param_distributions=param_grid, 
    cv=5, 
    n_iter = 50, 
    n_jobs = -1
) 
#замер времени
%time random_search.fit(X_train, y_train) 
#предсказание результата
y_test_pred = random_search.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(random_search.best_params_))

CPU times: total: 3.38 s
Wall time: 29.9 s
f1_score на тестовом наборе: 0.79
Наилучшие значения гиперпараметров: {'solver': 'saga', 'penalty': 'l1', 'C': 0.3}


результат тот же, что и с GridSearchCV

# <center> GridSearchCV + RandomForest

In [ ]:
#задаем параметры 
param_grid = {'n_estimators': list(range(80, 200, 30)), 
              'min_samples_leaf': [5,7],
              'max_depth': list(np.linspace(20, 30, 5, dtype=int))
              }
#создаем grid search со случайным лесом           
grid_search_forest = GridSearchCV(
    estimator=ensemble.RandomForestClassifier(random_state=42), 
    param_grid=param_grid, 
    cv=5, 
    n_jobs = -1
)  
#замер времени
%time grid_search_forest.fit(X_train, y_train) 
#предсказание результата
y_test_pred = grid_search_forest.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(grid_search_forest.best_params_))

CPU times: total: 2.8 s
Wall time: 35 s
f1_score на тестовом наборе: 0.81
Наилучшие значения гиперпараметров: {'max_depth': 20, 'min_samples_leaf': 5, 'n_estimators': 170}


В данном случае метрика выросла больше, чем с логистической регрессией, но не изменилась относительно модели, с дефолтными гиперпараметрами

# <center> RandomizedSearchCV + RandomForest

In [ ]:
#задаем параметры 
param_grid = {'n_estimators': list(range(80, 200, 30)), 
              'min_samples_leaf': [3,5],
              'max_depth': list(np.linspace(10, 20, 5, dtype=int))
              }
#создаем RandomizedSearch со случайным лесом
random_search = RandomizedSearchCV(
    estimator=ensemble.RandomForestClassifier(random_state=42), 
    param_distributions=param_grid, 
    cv=5, 
    n_iter = 50, 
    n_jobs = -1
) 
#замер времени
%time random_search.fit(X_train, y_train) 
#предсказание результата
y_test_pred = random_search.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(random_search.best_params_))

CPU times: total: 2.02 s
Wall time: 34.9 s
f1_score на тестовом наборе: 0.80
Наилучшие значения гиперпараметров: {'n_estimators': 110, 'min_samples_leaf': 3, 'max_depth': 15}


В данном случае метрика незначительно ухудшилась

# <center> Hyperopt + LogisticRegression

In [ ]:
#создаем пространство параметров
space= {
    'penalty': hp.choice('penalty', ['l2']), 
    'solver': hp.choice('solver', ['lbfgs','sag','liblinear','saga']), 
    'C': hp.uniform('C', 0.01, 1)
}

In [ ]:
# зафксируем random_state
random_state = 42
# функция hyperopt с линейной регрессией
def hyperopt_reg(params, cv=5, X=X_train, y=y_train, random_state=random_state):
    params = {'penalty': params['penalty'], 
              'solver': params['solver'], 
              'C': params['C']
              }
  
    
    model = linear_model.LogisticRegression(**params, random_state=random_state,max_iter=50)

    # обучаем модель
    model.fit(X, y)
    
    score = cross_val_score(model, X, y, cv=5, scoring="f1", n_jobs=-1).mean()


    return -score

In [ ]:
%%time
# начинаем подбор гиперпараметров

trials = Trials() # используется для логирования результатов
#алгоритм перебора параметров
best=fmin(hyperopt_reg,
          space=space,
          algo=tpe.suggest,
          max_evals=50,
          trials=trials,
          rstate=np.random.default_rng(random_state)
         )

print("Наилучшие значения гиперпараметров {}".format(best))

100%|██████████| 50/50 [01:52<00:00,  2.26s/trial, best loss: -0.7949085610506387]
Наилучшие значения гиперпараметров {'C': 0.017331323696411022, 'penalty': 0, 'solver': 0}
CPU times: total: 1min 12s
Wall time: 1min 52s


In [ ]:
# рассчитаем точность для тестовой выборки
model = linear_model.LogisticRegression(
    random_state=random_state, 
    penalty='l2',
    solver='lbfgs',
    C=0.017331323696411022
)
#обучаем модель
model.fit(X_train, y_train)
#предсказание результата
y_test_pred = model.predict(X_test)

print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.78


метрика немного ухудшилась относительно модели с дефолтными гиперпараметрами

Так как, не все алгоритмы оптимизации работают с L1 регуляризацией, отдельно проверим параметры с использованием L1

In [ ]:
#создаем пространство параметров
space= {
    'penalty': hp.choice('penalty', ['l1','l2']),  # Только один вариант
    'solver': hp.choice('solver', ['liblinear','saga']),  # Только один вариант
    'C': hp.uniform('C', 0.01, 1)
}

In [ ]:
%%time
# начинаем подбор гиперпараметров

trials = Trials() # используется для логирования результатов
#алгоритм перебора параметров
best=fmin(hyperopt_reg, 
          space=space, 
          algo=tpe.suggest, 
          max_evals=50, 
          trials=trials, 
          rstate=np.random.default_rng(random_state)
         )

print("Наилучшие значения гиперпараметров {}".format(best))

100%|██████████| 50/50 [02:55<00:00,  3.52s/trial, best loss: -0.7922640819759152]
Наилучшие значения гиперпараметров {'C': 0.27904611131273815, 'penalty': 0, 'solver': 1}
CPU times: total: 1min 22s
Wall time: 2min 55s


In [ ]:
# рассчитаем точность для тестовой выборки
model = linear_model.LogisticRegression(
    random_state=random_state, 
    penalty='l1',
    solver='saga',
    C=0.27904611131273815
)
#обучаем модель
model.fit(X_train, y_train)
#предсказание результата
y_test_pred = model.predict(X_test)

print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.78


Метрика такая же, что и с дефолтными гиперпараметрами

# <center> Hyperopt + RandomForest

In [ ]:
# зададим пространство поиска гиперпараметров
space={'n_estimators': hp.quniform('n_estimators', 100, 200, 1),
       'max_depth' : hp.quniform('max_depth', 15, 26, 1),
       'min_samples_leaf': hp.quniform('min_samples_leaf', 2, 10, 1)
      }

In [ ]:
def hyperopt_rf(params, cv=5, X=X_train, y=y_train, random_state=random_state):
    
    params = {'n_estimators': int(params['n_estimators']), 
              'max_depth': int(params['max_depth']), 
             'min_samples_leaf': int(params['min_samples_leaf'])
              }
  
    
    model = ensemble.RandomForestClassifier(**params, random_state=random_state)

    # обучаем модель
    model.fit(X, y)

    score = cross_val_score(model, X, y, cv=5, scoring="f1", n_jobs=-1).mean()

    
    return -score

In [ ]:
%%time
# начинаем подбор гиперпараметров

trials = Trials() # используется для логирования результатов

best=fmin(hyperopt_rf,
          space=space,
          algo=tpe.suggest, 
          max_evals=50,
          trials=trials, 
          rstate=np.random.default_rng(random_state)
         )

print("Наилучшие значения гиперпараметров {}".format(best))

100%|██████████| 50/50 [02:50<00:00,  3.41s/trial, best loss: -0.8190059974903507]
Наилучшие значения гиперпараметров {'max_depth': 15.0, 'min_samples_leaf': 2.0, 'n_estimators': 114.0}
CPU times: total: 1min 18s
Wall time: 2min 50s


In [ ]:
# рассчитаем точность для тестовой выборки
model = ensemble.RandomForestClassifier(
    random_state=random_state, 
    n_estimators=int(best['n_estimators']),
    max_depth=int(best['max_depth']),
    min_samples_leaf=int(best['min_samples_leaf'])
)
# обучаем модель
model.fit(X_train, y_train)
#предсказание результата
y_test_pred = model.predict(X_test)

print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.81


Метрика не отличается от метрики модели с дефолтными параметрами

# <center> Optuna + LogisticRegression

In [ ]:
#функция с алгоритмом optuna и линейной регрессией
def optuna_reg(trial):
  # задаем пространства поиска гиперпараметров
  penalty = trial.suggest_categorical('penalty', ['l2'])
  solver = trial.suggest_categorical('solver', ['lbfgs','sag','liblinear','saga'])
  C = trial.suggest_float('C', 0.01, 1)

  # создаем модель
  model = linear_model.LogisticRegression(penalty=penalty,
                                          solver=solver,
                                          C=C,
                                          random_state=random_state,
                                          max_iter=50)
  # обучаем модель
  model.fit(X_train, y_train)
  
  score = cross_val_score(model, X, y, cv=5, scoring="f1", n_jobs=-1).mean()

  return score

In [ ]:
%%time
# cоздаем объект исследования
# можем напрямую указать, что нам необходимо максимизировать метрику direction="maximize"
study = optuna.create_study(study_name="LogisticRegression", direction="maximize")
# ищем лучшую комбинацию гиперпараметров n_trials раз
study.optimize(optuna_reg, n_trials=50)

In [ ]:
# выводим результаты на обучающей выборке
print("Наилучшие значения гиперпараметров {}".format(study.best_params))

Наилучшие значения гиперпараметров {'penalty': 'l2', 'solver': 'sag', 'C': 0.027462308289988836}


In [ ]:
# рассчитаем точность для тестовой выборки
model = linear_model.LogisticRegression(**study.best_params,random_state=random_state, )
#обучаем модель
model.fit(X_train, y_train)
#предсказываем результат
y_test_pred = model.predict(X_test)

print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.78


Также нет отличий от дефолтных параметров

Так как, не все алгоритмы оптимизации работают с L1 регуляризацией, отдельно проверим параметры с использованием L1

In [ ]:
#функция с алгоритмом optuna и линейной регрессией
def optuna_reg(trial):
  # задаем пространства поиска гиперпараметров
  penalty = trial.suggest_categorical('penalty', ['l1','l2'])
  solver = trial.suggest_categorical('solver', ['liblinear','saga'])
  C = trial.suggest_float('C', 0.01, 1)

  # создаем модель
  model = linear_model.LogisticRegression(penalty=penalty,
                                          solver=solver,
                                          C=C,
                                          random_state=random_state,
                                          max_iter=50)
  # обучаем модель
  model.fit(X_train, y_train)
  score = cross_val_score(model, X, y, cv=5, scoring="f1", n_jobs=-1).mean()

  return score

In [ ]:
%%time
# cоздаем объект исследования
# можем напрямую указать, что нам необходимо максимизировать метрику direction="maximize"
study = optuna.create_study(study_name="LogisticRegression", direction="maximize")
# ищем лучшую комбинацию гиперпараметров n_trials раз
study.optimize(optuna_reg, n_trials=50)

[I 2025-12-25 21:33:35,320] A new study created in memory with name: LogisticRegression
[I 2025-12-25 21:33:36,367] Trial 0 finished with value: 0.7846489233754406 and parameters: {'penalty': 'l1', 'solver': 'liblinear', 'C': 0.3683691110412774}. Best is trial 0 with value: 0.7846489233754406.
[I 2025-12-25 21:33:37,709] Trial 1 finished with value: 0.7826515528863012 and parameters: {'penalty': 'l1', 'solver': 'liblinear', 'C': 0.5205385328701491}. Best is trial 0 with value: 0.7846489233754406.
[I 2025-12-25 21:33:43,431] Trial 2 finished with value: 0.7843375145765479 and parameters: {'penalty': 'l1', 'solver': 'saga', 'C': 0.8462919576742493}. Best is trial 0 with value: 0.7846489233754406.
[I 2025-12-25 21:33:44,473] Trial 3 finished with value: 0.7801309483153497 and parameters: {'penalty': 'l2', 'solver': 'liblinear', 'C': 0.20591309354353773}. Best is trial 0 with value: 0.7846489233754406.
[I 2025-12-25 21:33:48,433] Trial 4 finished with value: 0.7786091397511902 and paramete

CPU times: total: 39.6 s
Wall time: 1min 44s


In [ ]:
# выводим результаты на обучающей выборке
print("Наилучшие значения гиперпараметров {}".format(study.best_params))

Наилучшие значения гиперпараметров {'penalty': 'l2', 'solver': 'liblinear', 'C': 0.02794390329994935}


In [ ]:
# рассчитаем точность для тестовой выборки
model = linear_model.LogisticRegression(**study.best_params,random_state=random_state, )
#обучаем модель
model.fit(X_train, y_train)
#предсказываем результат
y_test_pred = model.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.78


Метрика упала относительно дефолтных параметров

# <center> Optuna + RandomForest

In [13]:
#функция с алгоритмом optuna и случайным лесом
def optuna_rf(trial):
  # задаем пространства поиска гиперпараметров
  n_estimators = trial.suggest_int('n_estimators', 100, 200, 1)
  max_depth = trial.suggest_int('max_depth', 10, 30, 1)
  min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 10, 1)

  # создаем модель
  model = ensemble.RandomForestClassifier(n_estimators=n_estimators,
                                          max_depth=max_depth,
                                          min_samples_leaf=min_samples_leaf,
                                          random_state=42)
  # обучаем модель
  model.fit(X_train, y_train)
  
  score = cross_val_score(model, X, y, cv=5, scoring="f1", n_jobs=-1).mean()

  return score

In [14]:
%%time
# cоздаем объект исследования
# можем напрямую указать, что нам необходимо максимизировать метрику direction="maximize"
study = optuna.create_study(study_name="RandomForestClassifier", direction="maximize")
# ищем лучшую комбинацию гиперпараметров n_trials раз
study.optimize(optuna_rf, n_trials=5)

[I 2026-01-29 18:35:46,916] A new study created in memory with name: RandomForestClassifier
[I 2026-01-29 18:35:51,382] Trial 0 finished with value: 0.8001579477946101 and parameters: {'n_estimators': 131, 'max_depth': 29, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.8001579477946101.
[I 2026-01-29 18:35:54,766] Trial 1 finished with value: 0.8001667693730425 and parameters: {'n_estimators': 103, 'max_depth': 13, 'min_samples_leaf': 6}. Best is trial 1 with value: 0.8001667693730425.
[I 2026-01-29 18:35:59,538] Trial 2 finished with value: 0.8051994912185101 and parameters: {'n_estimators': 176, 'max_depth': 26, 'min_samples_leaf': 7}. Best is trial 2 with value: 0.8051994912185101.
[I 2026-01-29 18:36:03,830] Trial 3 finished with value: 0.8019245020280852 and parameters: {'n_estimators': 163, 'max_depth': 11, 'min_samples_leaf': 5}. Best is trial 2 with value: 0.8051994912185101.
[I 2026-01-29 18:36:07,125] Trial 4 finished with value: 0.8059514717889892 and parameters: {'n_

CPU times: total: 6.84 s
Wall time: 20.2 s


In [15]:
# выводим результаты на обучающей выборке
print("Наилучшие значения гиперпараметров {}".format(study.best_params))


Наилучшие значения гиперпараметров {'n_estimators': 144, 'max_depth': 22, 'min_samples_leaf': 5}


In [ ]:
# рассчитаем точность для тестовой выборки
model = ensemble.RandomForestClassifier(**study.best_params,random_state=random_state, )
#обучаем модель
model.fit(X_train, y_train)
#предсказание результата
y_test_pred = model.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

f1_score на тестовом наборе: 0.80


Метрика незначительно уменьшилась относительно дефолтных праметров

# Выводы: самый высокий рузльтат F1 метрики наблюдается у дефолтной модели случайного леса (RandomForest), другие модели справились либо также, либо хуже при том, что заняли много времени.